# Part 2 — Repeated-event count models

In [1]:
from math import comb
from scipy.stats import binom, poisson

## Late supplier deliveries: binomial

> A project expects 30 deliveries. Each is modelled as late with probability 0.12 independently of
> the others. Let $B$ be the number of late deliveries.

### a)

> Justify $B \sim \text{Bin}(30, 0.12)$. Identify the Bernoulli trial and discuss fixed $n$, constant
> $p$, independence, and one realistic violation.

**Bernoulli trial:** one delivery. It has two possible outcomes: late, counted as a "success" with
probability $p = 0.12$, or on time, a "failure" with probability $1 - p = 0.88$.

**Why the binomial model fits:** $B$ counts the successes in 30 Bernoulli trials, so it is binomial
if three conditions hold:

- **Fixed $n$:** the project has a set number of deliveries, $n = 30$, known before any arrive.
- **Constant $p$:** every delivery has the same probability 0.12 of being late.
- **Independence:** whether one delivery is late does not change the chance that another is late.

This gives $B \sim \text{Bin}(30, 0.12)$ with parameters $n = 30$ and $p = 0.12$, and support
$\{0, 1, 2, \dots, 30\}$.

**One realistic violation:** late deliveries tend to come in clusters. A storm, for example, can
delay several deliveries at once. One late delivery then makes the
next more likely to be late, so the deliveries are not independent and $p$ is not constant.

### b)

> Use SciPy to calculate $P(B = 3)$, $P(B \le 4)$, and $P(B \ge 5)$. For $P(B = 3)$, also show the
> binomial formula and explain $\binom{n}{k}$. For the upper tail, explain the correct
> survival-function argument.

**Binomial formula:**

$$P(B = k) = \binom{n}{k} p^k (1 - p)^{n - k}$$

$$P(B = 3) = \binom{30}{3} (0.12)^3 (0.88)^{27}$$

**What $\binom{n}{k}$ means:** the number of ways to choose which $k$ of the $n$ trials are the
successes. Each way has probability $(0.12)^3 (0.88)^{27}$, and there are

$$\binom{30}{3} = \frac{30!}{3!\,27!} = \frac{30 \times 29 \times 28}{3 \times 2 \times 1} = 4060$$

such ways, so that probability is multiplied by 4060.

In [2]:
B = binom(n=30, p=0.12)

print(f"P(B = 3)  = {B.pmf(3):.4f}")
print(f"P(B <= 4) = {B.cdf(4):.4f}")
print(f"P(B >= 5) = {B.sf(4):.4f}")   # sf(4) = P(B > 4), which is the same as P(B >= 5)

# The binomial formula by hand gives the same P(B = 3):
print(f"\nC(30, 3) = {comb(30, 3)}")
print(f"C(30, 3) * 0.12^3 * 0.88^27 = {comb(30, 3) * 0.12**3 * 0.88**27:.4f}")

P(B = 3)  = 0.2224
P(B <= 4) = 0.7118
P(B >= 5) = 0.2882

C(30, 3) = 4060
C(30, 3) * 0.12^3 * 0.88^27 = 0.2224


**Survival-function argument:** SciPy's survival function is $\text{sf}(k) = P(B > k)$. Because $B$
only takes whole numbers, "at least 5" is the same as "more than 4":

$$P(B \ge 5) = P(B > 4) = \text{sf}(4) = 1 - P(B \le 4)$$

So the correct argument is 4: $\text{sf}(5)$ would give $P(B \ge 6)$ and leave out $B = 5$.

**Interpretation:** 71 % of projects have at most four late deliveries, and 29 % have five or more.

### c)

> Calculate and interpret the mean and standard deviation.

$$E[B] = np, \qquad \text{SD}(B) = \sqrt{np(1 - p)}$$

In [3]:
print(f"E[B]  = 30 * 0.12 = {B.mean():.2f}")
print(f"SD(B) = sqrt(30 * 0.12 * 0.88) = {B.std():.4f}")

E[B]  = 30 * 0.12 = 3.60
SD(B) = sqrt(30 * 0.12 * 0.88) = 1.7799


**$E[B] = 3.6$:** the long-run average number of late deliveries per project.

**$\text{SD}(B) = 1.7799$:** the number of late deliveries typically differs from 3.6 by about 1.8.

## Production stoppages: Poisson

> Short stoppages are modelled as a Poisson process with mean 1.6 per 8-hour shift. Let $Y$ be the
> count during one shift.

### a)

> State the model, parameter, support, mean, and variance. Calculate $P(Y = 0)$ and $P(Y \ge 3)$ for
> one shift.

**Model:** $Y \sim \text{Pois}(\lambda)$, with PMF

$$P(Y = y) = \frac{e^{-\lambda} \lambda^y}{y!}$$

**Parameter:** $\lambda = 1.6$, the expected number of stoppages in one 8-hour shift. In SciPy it is
passed as `mu`.

**Support:** $\{0, 1, 2, \dots\}$.

**Mean and variance:**

$$E[Y] = \text{Var}(Y) = \lambda = 1.6$$

**Probabilities:**

$$P(Y = 0) = \frac{e^{-1.6}\, 1.6^0}{0!} = e^{-1.6}, \qquad
  P(Y \ge 3) = P(Y > 2) = \text{sf}(2)$$

In [4]:
Y = poisson(mu=1.6)

print(f"E[Y] = {Y.mean():.1f},  Var(Y) = {Y.var():.1f}")
print(f"P(Y = 0)  = {Y.pmf(0):.4f}")
print(f"P(Y >= 3) = {Y.sf(2):.4f}")   # sf(2) = P(Y > 2), which is the same as P(Y >= 3)

E[Y] = 1.6,  Var(Y) = 1.6
P(Y = 0)  = 0.2019
P(Y >= 3) = 0.2166


**Interpretation:** 20 % of shifts have no stoppages at all, and 22 % have three or more.

### b)

> Scale the expected count and calculate the probability of at least three stoppages in 24 hours.
> Discuss the stable-rate and independent-event assumptions.

Let $Y_{24}$ be the number of stoppages in 24 hours. Since 24 hours is three 8-hour shifts, the
expected count is multiplied by 3:

$$\lambda_{24} = 3 \times 1.6 = 4.8, \qquad Y_{24} \sim \text{Pois}(4.8)$$

In [5]:
Y_24 = poisson(mu=3 * 1.6)

print(f"lambda_24    = {Y_24.mean():.1f}")
print(f"P(Y_24 >= 3) = {Y_24.sf(2):.4f}")   # sf(2) = P(Y_24 > 2), the same as P(Y_24 >= 3)

lambda_24    = 4.8
P(Y_24 >= 3) = 0.8575


**Interpretation:** 86 % of days have at least three stoppages.

**Stable-rate assumption:** multiplying by 3 assumes the rate of 1.6 per 8 hours holds in every
shift of the day. If the night shift has fewer stoppages, for example, the true expected count is
below 4.8 and the 86 % is too high.

**Independent-event assumption:** one stoppage must not change the chance of another. In practice
stoppages often come in bursts, for example when restarting the line after a stoppage causes another
jam soon after.